In [2]:
import warnings
warnings.filterwarnings("ignore")
# Import PDF loader
#!pip install pymupdf
from langchain_community.document_loaders import PyMuPDFLoader, TextLoader
#import fitz ## FOr PyMuPDF
#from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain.vectorstores import  FAISS
from langchain_core.vectorstores import VectorStoreRetriever
from langchain.chains import  RetrievalQA
from langchain_openai import ChatOpenAI
from langchain_community.chat_models import ChatOllama
## Define class for description of inputs in structured tool 
import os
import subprocess
#from langchain.pydantic_v1 import BaseModel, Field
from pydantic import BaseModel, Field
from langchain.tools import BaseTool, StructuredTool, tool
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor,create_react_agent,create_openai_functions_agent # To load simple ReAct agent. Reason an act
from langchain import hub

import os 
import dotenv
dotenv.load_dotenv()

# Defien OPENAI API
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")


In [5]:
# Define Embedding Model
from langchain_openai import OpenAIEmbeddings

# Load embeddings
embeddings = OpenAIEmbeddings()

# Define name of PDF
pdf_filename = "./Equi_GNN.pdf"

# Load the PDF with PyMuPDF
pdf_loader = PyMuPDFLoader(pdf_filename)

# Load documents
documents = pdf_loader.load()

# ##Get the embeddings of the documents in the FAISS vector store
# ## NOTE: THis has to be ran one time only to generate the embeddings. If the embeddings are locally saved then this step can be skipped, and the vectorbase can be upload the with the code bellow. 
# doc_embeddings = FAISS.from_documents(documents, embeddings)

## Save the vector store locally 
name = "./knowledge_base/equi_gnn/"
#doc_embeddings.save_local(name)


## Load the vector store
doc_embeddings = FAISS.load_local(name, embeddings,allow_dangerous_deserialization=True)



MuPDF error: syntax error: could not parse color space (1467 0 R)

MuPDF error: syntax error: could not parse color space (1748 0 R)



In [6]:
# Defin etest Queries
query_1 = "What is equivariance in terms of Neural Networks?"

q1_answer = doc_embeddings.similarity_search_with_score(query_1)
q1_answer[0]



(Document(page_content='5\nEquivariant Geometric GNNs\nOverview. As explained in the previous section, invariant GNNs pre-compute a set of local invariants\nin each neighbourhood before performing message passing. While this can be efficient, it is also\nrestrictive. A key limitation of invariant GNNs is that the set of local invariants is fixed and has to be\ndetermined prior to message passing.\nHow could we overcome this limitation and instead allow the network to learn its set of invariants\nwhich could (1) be more suited to the task at hand, and (2) whose complexity could be controlled by\nperforming more or fewer message passing steps?\nIn this section, we investigate a family of GNNs, which we will refer to as equivariant GNNs\n(EGNNs), that fulfil these two goals and additionally allow the prediction of equivariant quantities.\nThese models do not pre-compute local invariants but instead perform message passing in a way such\nthat the hidden features at each layer are equivaria

In [8]:
## Define retriever
retreiver = doc_embeddings.as_retriever()

# Define llm
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

#llm = ChatOllama(model="gemma:2b",temperature=0.)

# Define Chain type 
chain_type = "stuff"

# Define the QA model
qa_model = RetrievalQA.from_chain_type(llm=llm, retriever=retreiver, chain_type=chain_type)

# Define New question for retreiver
new_question = "What is the difference between equivariance and invariance in Neural Networks?"

# Get the answer
answer = qa_model.invoke(new_question)

print(answer['result'])

In the context of Neural Networks, particularly in Geometric Graph Neural Networks (GNNs), the concepts of equivariance and invariance refer to how the network's outputs respond to transformations of the input data.

1. **Invariance**: A neural network is said to be invariant to a transformation if the output remains unchanged when the input undergoes that transformation. For example, if a model is invariant to rotation, rotating the input data will not affect the output of the model. Invariant models pre-compute a set of local invariants before performing message passing, which means they have a fixed set of features that do not change regardless of the input transformations.

2. **Equivariance**: In contrast, a neural network is equivariant to a transformation if the output changes in a predictable way when the input is transformed. Specifically, if the input is transformed (e.g., rotated), the output will also be transformed in the same way. This means that the hidden features at ea

### Tool 1: RAG_gen_database_tool

This tool can be used to process a PDF and generate a vector store using OpenAI embeddings and LLMs, And FAISS for storage and similarity search. 



In [10]:
## Define class for description of inputs in structured tool
class RAG_Gen_DB_Inputs(BaseModel):
    pdf_filename: str = Field(description="Path to the PDF file to be embedded")
    name: str = Field(description="Name of the directory to save the vector store")

## Define fucntion to generate vectorestore from PDF file
def generate_vectorstore_from_pdf(pdf_filename, name):
    # Load the PDF with PyMuPDF
    pdf_loader = PyMuPDFLoader(pdf_filename)

    # Load documents
    documents = pdf_loader.load()

    # Define Embedding Model
    from langchain_openai import OpenAIEmbeddings

    # Load embeddings
    embeddings = OpenAIEmbeddings()

    ##Get the embeddings of the documents in the FAISS vector store
    doc_embeddings = FAISS.from_documents(documents, embeddings)

    ## Save the vector store locally 
    doc_embeddings.save_local('knowledge_base/'+name)

    return

# # Define name of PDF
# pdf_filename = "./formation-mech-SBU-Cr-BDC-MOF.pdf"
# ## Test the function 
# generate_vectorstore_from_pdf(pdf_filename, "SBU_form",)

## Define Structured Tool
RAG_DB_gen_tool = StructuredTool.from_function(
    func=generate_vectorstore_from_pdf, # Function to be used
    name="RAG_DB_gen_tool", # Name of the tool
    description="Generate a vector store from a PDF file. First embedthe data and stores it for later usage.", # Description of the tool
    args_schema=RAG_Gen_DB_Inputs, # Input schema
    return_direct=False, # Return the output directly
    handle_error=True, # Handle errors
    # Use dictionary as input
    )



## Define LLM
# llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.0)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

# Define list of tools the LLM is going to use 
tools = [RAG_DB_gen_tool]

## Propomt for openai function
prompt = hub.pull("hwchase17/openai-functions-agent")
#print(prompt)

# Create OpenAI functions agent
agent = create_openai_functions_agent(llm=llm, tools=tools, prompt=prompt)

# ## Create Agent executor
agent_executor = AgentExecutor(agent=agent,tools=tools,verbose=True,handle_parsing_errors=True)

def RAG_DB_gen_response(input_text:str):
    return agent_executor.invoke({"input": input_text})['output']

## Define test prompt 
test_prompt_1 = "Generate a vector store from the PDF file 'formation-mech-SBU-Cr-BDC-MOF.pdf' and save it as 'SBU_form'."

# Create and move to tool_1 directory
# os.chdir("tool_5")
print(RAG_DB_gen_response(test_prompt_1))
# os.chdir("..")




> Entering new AgentExecutor chain...

Invoking: `RAG_DB_gen_tool` with `{'pdf_filename': 'formation-mech-SBU-Cr-BDC-MOF.pdf', 'name': 'SBU_form'}`


NoneThe vector store has been successfully generated from the PDF file 'formation-mech-SBU-Cr-BDC-MOF.pdf' and saved as 'SBU_form'. If you need any further assistance, feel free to ask!

> Finished chain.
The vector store has been successfully generated from the PDF file 'formation-mech-SBU-Cr-BDC-MOF.pdf' and saved as 'SBU_form'. If you need any further assistance, feel free to ask!


## Tool 2: RAG_retreival_tool

This tool shas acces to a vectorstore database and is used for Retreival Augmented Generation (RAG). THe chatbot uses similarity search to find the most simiilar piece of information related to the input prompt, learns from it and generates an answer. 



In [12]:
## Define class for description of inputs in structured tool
class RAG_Retreive_DB_Inputs(BaseModel):
    question: str = Field(description="Question to be asked to the vector store")
    name: str = Field(description="Name of the directory to save the vector store")


## Define function to get the answer
def RAG_retreiver(question, name):

    # Load embeddings
    embeddings = OpenAIEmbeddings()

    ## Rename the name variable adding the path to general knowledge base
    name = "./knowledge_base/"+name
    ## Load the vector store
    doc_embeddings = FAISS.load_local(name, embeddings,allow_dangerous_deserialization=True)

    ## Define retriever
    retreiver = doc_embeddings.as_retriever()

    # Define llm
    llm = ChatOpenAI()
    #llm = ChatOllama(model="gemma:2b",temperature=0.)

    # Define Chain type 
    chain_type = "stuff"

    # Define the QA model
    qa_model = RetrievalQA.from_chain_type(llm=llm, retriever=retreiver, chain_type=chain_type)

    answer = qa_model.invoke(question)
    return print(answer['result'])

# ## Test the function
# q_1 = "What are the main conclussions in this paper?"
# name = "knowledge_base/SBU_form"
# RAG_retreiver(q_1, name)

## Define Structured Tool
RAG_Retreive_DB_tool = StructuredTool.from_function(
    func=RAG_retreiver, # Function to be used
    name="RAG_Retreive_DB_tool", # Name of the tool
    description="Retrieve information from a vector store.", # Description of the tool
    args_schema=RAG_Retreive_DB_Inputs, # Input schema
    return_direct=True, # Return the output directly
    handle_error=True, # Handle errors
    # Use dictionary as input
    )



## Define LLM
# llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.0)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

# Define list of tools the LLM is going to use 
tools = [RAG_Retreive_DB_tool]

## Propomt for openai function
prompt = hub.pull("hwchase17/openai-functions-agent")
#print(prompt)

# Create OpenAI functions agent
agent = create_openai_functions_agent(llm=llm, tools=tools, prompt=prompt)

# ## Create Agent executor
agent_executor = AgentExecutor(agent=agent,tools=tools,verbose=True,handle_parsing_errors=True)

def RAG_retreiver_response(input_text:str):
    return agent_executor.invoke({"input": input_text})['output']

## Define test prompt 
test_prompt_2 = "What are the main conclussions in this paper?. The vector store is called 'SBU_form'."
print(RAG_retreiver_response(test_prompt_2))




> Entering new AgentExecutor chain...

Invoking: `RAG_Retreive_DB_tool` with `{'question': 'What are the main conclusions in this paper?', 'name': 'SBU_form'}`


The main conclusions of this paper, which outlines the formation of the secondary building unit (SBU) of MIL-101, a chromium terephthalate metal-organic framework, are as follows:
1. The study proposes a detailed mechanism involving six reactions leading to the formation of the SBU in MIL-101, a crucial step for MOF nucleation.
2. The energy barriers for each reaction indicate that all products are thermodynamically favored over reactants, and the reactions are exothermic.
3. The formation of the metal core and SBU is found to be the rate-limiting step in MOF synthesis, as the high to low spin transition during metal complexation plays a critical role.
4. The proposed reaction series involves gradual linker addition to the metal core, with specific isomers and structural rearrangements influencing the pathway.
5. Kinetic mod